# Orçamento SUS BR
Análise dos gastos federais na função Saúde (código 10) entre 2015 e 2024.

## Objetivo

Investigar se o governo federal executa o orçamento planejado para a saúde pública, analisando os valores empenhados, liquidados e pagos ao longo dos anos.

## Fonte dos Dados

- **API:** Portal da Transparência
- **Endpoint:** `/api-de-dados/despesas/por-funcional-programatica`
- **Filtro:** Função orçamentária 10 - Saúde
- **Período:** 2015 a 2024

## Imports e Configurações
Carregamento das bibliotecas e variáveis de ambiente necessárias para a coleta.

In [6]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("API_KEY")

headers = {
    "chave-api-dados": API_KEY,
    "accept": "application/json"
}


## Extract
Coleta dos dados da API do Portal da Transparência ano a ano com paginação automática.

### Configuração da Requisição
Define os anos consultados, a URL do endpoint e os parâmetros de filtro.

In [9]:

anos = list(range(2015, 2025))
url = "https://api.portaldatransparencia.gov.br/api-de-dados/despesas/por-funcional-programatica"
dados = []

### Coleta dos Dados
Realiza a requisição à API ano a ano com paginação automática.
A coleta para quando a API retorna lista vazia.

In [51]:
for ano in anos:
    pagina = 1
    while True:
        params = {
            "ano": ano,
            "funcao": 10,
            "pagina": pagina
        }

        response = requests.get(
            url,
            headers=headers,
            params=params
        )

        resultado = response.json()

        if not resultado:
            break

        dados.extend(resultado)
        pagina +=1

    print(f"{ano} (X)")

df = pd.DataFrame(dados)

2015 (X)
2016 (X)
2017 (X)
2018 (X)
2019 (X)
2020 (X)
2021 (X)
2022 (X)
2023 (X)
2024 (X)


### Verificação dos Dados Brutos
Checagem inicial dos dados coletados antes do tratamento.

In [46]:
print(f"Número total de linhas = {df.shape[0]} e de colunas = {df.shape[1]}")

Número total de linhas = 3323 e de colunas = 12


In [47]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3323 entries, 0 to 3322
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ano              3323 non-null   int64 
 1   funcao           3323 non-null   object
 2   codigoFuncao     3323 non-null   object
 3   subfuncao        3323 non-null   object
 4   codigoSubfuncao  3323 non-null   object
 5   programa         3323 non-null   object
 6   codigoPrograma   3323 non-null   object
 7   acao             3323 non-null   object
 8   codigoAcao       3323 non-null   object
 9   empenhado        3323 non-null   object
 10  liquidado        3323 non-null   object
 11  pago             3323 non-null   object
dtypes: int64(1), object(11)
memory usage: 311.7+ KB
None


In [50]:
df.describe(exclude='number')


,funcao,codigoFuncao,subfuncao,codigoSubfuncao,programa,codigoPrograma,acao,codigoAcao,empenhado,liquidado,pago
count,3323,3323,3323,3323,3323,3323,3323,3323,3323,3323,3323
unique,1,1,32,32,77,71,384,332,1123,1064,1054
top,Saúde,10,Assistência hospitalar e ambulatorial,302,FORTALECIMENTO DO SISTEMA UNICO DE SAUDE (SUS),2015,ESTRUTURACAO DE UNIDADES DE ATENCAO ESPECIALIZ...,8535,"0,00","0,00","0,00"
freq,3323,3323,493,493,1103,1103,39,39,2185,2254,2265


## Transform
Tratamento dos dados brutos:
- Conversão dos valores financeiros de string para float

In [53]:
for col in ["empenhado", "liquidado", "pago"]:
    df[col] = df[col].str.replace(".", "")
    df[col] = df[col].str.replace(",", ".")
    df[col] = pd.to_numeric(df[col])

### Verificação após Transform
Checagem dos dados após o tratamento para garantir que os tipos estão corretos.

In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6646 entries, 0 to 6645
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ano              6646 non-null   int64  
 1   funcao           6646 non-null   object 
 2   codigoFuncao     6646 non-null   object 
 3   subfuncao        6646 non-null   object 
 4   codigoSubfuncao  6646 non-null   object 
 5   programa         6646 non-null   object 
 6   codigoPrograma   6646 non-null   object 
 7   acao             6646 non-null   object 
 8   codigoAcao       6646 non-null   object 
 9   empenhado        6646 non-null   float64
 10  liquidado        6646 non-null   float64
 11  pago             6646 non-null   float64
dtypes: float64(3), int64(1), object(8)
memory usage: 623.2+ KB


## Load
Exportação do dado tratado para CSV.
Destino: `data/processed/saude_orcamento.csv`

In [63]:
df.to_csv("../data/processed/saude_orcamento.csv", index=False)